# HEMA FAIR OMOP ETL
Creates a PostgreSQL OMOP CDM database, imports vocabulary files, and maps HEMA source data to OMOP tables.

## Connection Settings

In [ ]:
DB_HOST     = "localhost"
DB_PORT     = 5432
DB_NAME     = "trainee_01"   # replace with your trainee number
DB_USER     = "trainee_01"   # replace with your trainee number
DB_PASSWORD = ""             # fill in your password

print(f"Connecting to {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

## Setup

In [ ]:
import psycopg2
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

con = psycopg2.connect(
    host=DB_HOST, port=DB_PORT,
    dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD
)
con.autocommit = True

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

def execute(sql):
    with con.cursor() as cur:
        cur.execute(sql)

def query(sql):
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn)

print("Connected to PostgreSQL.")

## OMOP CDM Tables

In [ ]:
query("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'omop'
ORDER BY table_name
""")

## Fill CDM Source

In [ ]:
execute("""
INSERT INTO omop.cdm_source (
    cdm_source_name, cdm_source_abbreviation, cdm_holder,
    source_release_date, cdm_release_date,
    cdm_version_concept_id, vocabulary_version
)
VALUES (
    'HEMA', 'HEMA', 'HEMA',
    '2026-05-06', '2026-05-06',
    756265,  -- CDM version 5.4
    'v5.0 27-FEB-25'
);
""")

query("SELECT * FROM omop.cdm_source")

## Import HEMA Source Data

In [ ]:
execute("CREATE SCHEMA IF NOT EXISTS import")

df_dictionary = pd.read_csv('synthetic_DataDictionary_2026-05-04.csv')
df_dictionary.to_sql('dictionary', engine, schema='import', if_exists='replace', index=False)

df_mappings = pd.read_csv('synthetic_DataDictionary_OMOPMappings_2026-05-04 (1).csv')
df_mappings.to_sql('mappings', engine, schema='import', if_exists='replace', index=False)

df_labels = pd.read_csv('synthetic_output_labels_balanced_transfusion_gaussianv3.csv',
                        parse_dates=["Patient's date of birth"])
df_labels.to_sql('labels', engine, schema='import', if_exists='replace', index=False)

print("Source data imported.")

In [ ]:
print("--- dictionary ---")
display(query("SELECT * FROM import.dictionary LIMIT 5"))

print("--- mappings ---")
display(query("SELECT * FROM import.mappings LIMIT 5"))

print("--- labels ---")
display(query("SELECT * FROM import.labels LIMIT 5"))

## ETL: Fill OMOP Tables
### PERSON
PostgreSQL uses `EXTRACT` instead of DuckDB's `year()` / `month()` / `day()` shorthand.

In [ ]:
execute("""
INSERT INTO omop.person (
    person_id, gender_concept_id,
    year_of_birth, month_of_birth, day_of_birth,
    race_concept_id, ethnicity_concept_id,
    person_source_value, gender_source_value
)
SELECT
    row_number() OVER () AS person_id,
    CASE "Patient's sex at birth"
        WHEN 'Male'   THEN 8507
        WHEN 'Female' THEN 8532
        ELSE 0
    END,
    EXTRACT(YEAR  FROM "Patient's date of birth")::integer,
    EXTRACT(MONTH FROM "Patient's date of birth")::integer,
    EXTRACT(DAY   FROM "Patient's date of birth")::integer,
    0,
    0,
    "Record ID",
    "Patient's sex at birth"
FROM import.labels;
""")

query("SELECT * FROM omop.person")

### OBSERVATION_PERIOD
No recording date available — all data is treated as measured on 2026-05-12.

In [ ]:
execute("""
INSERT INTO omop.observation_period (
    observation_period_id, person_id,
    observation_period_start_date, observation_period_end_date,
    period_type_concept_id
)
SELECT
    row_number() OVER (),
    person_id,
    '2026-05-12',
    '2026-05-12',
    32809  -- Case Report Form
FROM omop.person;
""")

query("SELECT * FROM omop.observation_period")

### VISIT_OCCURRENCE

In [ ]:
execute("""
INSERT INTO omop.visit_occurrence (
    visit_occurrence_id, person_id, visit_concept_id,
    visit_start_date, visit_start_datetime,
    visit_end_date, visit_end_datetime,
    visit_type_concept_id
)
SELECT
    row_number() OVER (),
    p.person_id,
    581476,  -- Home visit
    op.observation_period_start_date,
    op.observation_period_start_date,
    op.observation_period_end_date,
    op.observation_period_end_date,
    32809  -- Case report form
FROM omop.person p
JOIN omop.observation_period op ON op.person_id = p.person_id;
""")

query("SELECT * FROM omop.visit_occurrence")

### CONDITION_OCCURRENCE
Sequences generate continuous IDs across multiple inserts.

In [ ]:
execute("CREATE SEQUENCE condition_id_seq START 1;")

# Primary diagnosis
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT
    nextval('condition_id_seq'),
    p.person_id,
    CASE "Diagnosis retained by the specialised centre"
        WHEN 'Alpha-thalassemia'   THEN 4287844
        WHEN 'Beta-thalassemia'    THEN 65959000
        WHEN 'Sickle cell disease' THEN 25518  -- Sickle cell trait concept
        ELSE 0
    END,
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865,  -- Patient self report
    vo.visit_occurrence_id,
    "Diagnosis retained by the specialised centre"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

query("SELECT * FROM omop.condition_occurrence")

In [ ]:
# Hypopituitarism (no results expected — no 'Yes' values in synthetic data)
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    4254542,  -- Hypopituitarism
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Hypopituitarism"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Hypopituitarism" = 'Yes';
""")

# Infertile
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    4311387,  -- Infertile
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Is the patient infertile?"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Is the patient infertile?" = 'Yes';
""")

# Acute viral hepatitis (no results expected in synthetic data)
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    4211974,  -- Acute viral hepatitis
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Acute viral hepatitis"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Acute viral hepatitis" = 'Yes';
""")

# Vitamin D deficiency
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    436070,  -- Vitamin D deficiency
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Vitamin D deficiency"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Vitamin D deficiency" = 'Yes, confirmed';
""")

# Osteoporosis
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    80502,  -- Osteoporosis
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Osteoporosis"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Osteoporosis" = 'Yes, confirmed';
""")

# Osteopenia
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    4195039,  -- Osteopenia
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Osteopenia"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Osteopenia" = 'Yes, confirmed';
""")

# Acute chest syndrome
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    254062,  -- Acute chest syndrome
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Acute chest syndrome"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Acute chest syndrome" = 'Yes, confirmed';
""")

query("SELECT * FROM omop.condition_occurrence")

### OBSERVATION

In [ ]:
execute("CREATE SEQUENCE observation_id_seq START 1;")

# Country of birth
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, visit_occurrence_id, value_source_value
)
SELECT
    nextval('observation_id_seq'), p.person_id,
    CASE l."Patient's country of birth"
        WHEN 'Australia'      THEN 4199969
        WHEN 'Canada'         THEN 4200105
        WHEN 'Congo'          THEN 4202085
        WHEN 'Cyprus'         THEN 4152209
        WHEN 'Greece'         THEN 4151604
        WHEN 'Iraq'           THEN 4152215
        WHEN 'Syria'          THEN 4153306
        WHEN 'United Kingdom' THEN 4202086  -- England
        ELSE 40482029  -- Country of birth unknown
    END,
    vo.visit_start_date, vo.visit_start_date,
    32865,  -- Patient self report
    vo.visit_occurrence_id,
    l."Patient's country of birth"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

# Transfusion status
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, visit_occurrence_id, value_source_value
)
SELECT
    nextval('observation_id_seq'), p.person_id,
    40758326,  -- Transfusion status qualitative
    vo.visit_start_date, vo.visit_start_date, 32865, vo.visit_occurrence_id,
    l."Does the patient require regular or occasional transfusions in the present (in the last 12 months) ?"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

# HBB Allele 1 (not represented in OHDSI vocabularies)
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, value_as_string,
    visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    0,  -- Not represented in OHDSI Standardized Vocabularies
    vo.visit_start_date, vo.visit_start_date, 32865,
    l."HBB Allele 1", vo.visit_occurrence_id,
    'HGNC:4827'  -- From mappings file
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

# HBB Allele 2
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, value_as_string,
    visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    0, vo.visit_start_date, vo.visit_start_date, 32865,
    l."HBB Allele 2", vo.visit_occurrence_id, 'HGNC:4827'
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

# HBA Allele 1
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, value_as_string,
    visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    0, vo.visit_start_date, vo.visit_start_date, 32865,
    l."HBA Allele 1", vo.visit_occurrence_id, 'HGNC:4823;HGNC:4824'
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

# HBA Allele 2
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, value_as_string,
    visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    0, vo.visit_start_date, vo.visit_start_date, 32865,
    l."HBA Allele 2", vo.visit_occurrence_id, 'HGNC:HGNC:4823;HGNC:4824'
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

# Drug compliance — poor
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    4292063,  -- Drug compliance poor
    vo.visit_start_date, vo.visit_start_date, 32865,
    vo.visit_occurrence_id, l."Drug compliance"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Drug compliance" = 'Poor (< 60% of recommended dose)';
""")

# Drug compliance — good / excellent
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    4056965,  -- Drug compliance good
    vo.visit_start_date, vo.visit_start_date, 32865,
    vo.visit_occurrence_id, l."Drug compliance"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Drug compliance" = 'Good (80-60 % of recommended dose)'
   OR l."Drug compliance" = 'Excellent (>80% of recommended dose)';
""")

query("SELECT * FROM omop.observation")

### MEASUREMENT

In [ ]:
execute("CREATE SEQUENCE measurement_id_seq START 1;")

# Cardiac iron T2* (ms) — no OMOP concept available
execute("""
INSERT INTO omop.measurement (
    measurement_id, person_id, measurement_concept_id,
    measurement_date, measurement_datetime,
    measurement_type_concept_id, value_as_number,
    unit_concept_id, visit_occurrence_id, value_source_value
)
SELECT nextval('measurement_id_seq'), p.person_id,
    0,  -- No suitable mapping
    vo.visit_start_date, vo.visit_start_date, 32809,
    l."Cardiac iron T2*  (milisec)",
    9593,  -- millisecond
    vo.visit_occurrence_id, 'Cardiac iron T2'
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Cardiac iron T2*  (milisec)" IS NOT NULL;
""")

# Liver MRI T2* (ms)
execute("""
INSERT INTO omop.measurement (
    measurement_id, person_id, measurement_concept_id,
    measurement_date, measurement_datetime,
    measurement_type_concept_id, value_as_number,
    unit_concept_id, visit_occurrence_id, value_source_value
)
SELECT nextval('measurement_id_seq'), p.person_id,
    0,  -- No suitable mapping
    vo.visit_start_date, vo.visit_start_date, 32809,
    l."Liver MRI T2* (milisec)",
    9593, vo.visit_occurrence_id, 'Liver MRI T2'
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Liver MRI T2* (milisec)" IS NOT NULL;
""")

# Cirrhosis
execute("""
INSERT INTO omop.measurement (
    measurement_id, person_id, measurement_concept_id,
    measurement_date, measurement_datetime,
    measurement_type_concept_id, unit_concept_id,
    visit_occurrence_id, value_source_value
)
SELECT nextval('measurement_id_seq'), p.person_id,
    36770062,  -- Cirrhosis
    vo.visit_start_date, vo.visit_start_date, 32809,
    9593, vo.visit_occurrence_id, l."Cirrhosis"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Cirrhosis" = 'Yes, confirmed';
""")

# Ferritin serum (ng/mL)
execute("""
INSERT INTO omop.measurement (
    measurement_id, person_id, measurement_concept_id,
    measurement_date, measurement_datetime,
    measurement_type_concept_id, value_as_number,
    unit_concept_id, visit_occurrence_id
)
SELECT nextval('measurement_id_seq'), p.person_id,
    37208753,  -- Ferritin mass concentration in serum
    vo.visit_start_date, vo.visit_start_date, 32809,
    l."Ferritin serum (ng/mL / µg/L)",
    8842,  -- ng/mL
    vo.visit_occurrence_id
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Ferritin serum (ng/mL / µg/L)" IS NOT NULL;
""")

# Serum iron (μg/dL)
execute("""
INSERT INTO omop.measurement (
    measurement_id, person_id, measurement_concept_id,
    measurement_date, measurement_datetime,
    measurement_type_concept_id, value_as_number,
    unit_concept_id, visit_occurrence_id
)
SELECT nextval('measurement_id_seq'), p.person_id,
    4097596,  -- Serum iron measurement
    vo.visit_start_date, vo.visit_start_date, 32809,
    l."Serum iron (μg/dL)",
    8837,  -- ug/dL
    vo.visit_occurrence_id
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Serum iron (μg/dL)" IS NOT NULL;
""")

query("SELECT * FROM omop.measurement")

### PROCEDURE_OCCURRENCE

In [ ]:
execute("CREATE SEQUENCE procedure_id_seq START 1;")

# Chelation treatment
execute("""
INSERT INTO omop.procedure_occurrence (
    procedure_occurrence_id, person_id, procedure_concept_id,
    procedure_date, procedure_datetime,
    procedure_end_date, procedure_end_datetime,
    procedure_type_concept_id, visit_occurrence_id, procedure_source_value
)
SELECT nextval('procedure_id_seq'), p.person_id,
    4068544,  -- Chelation (of cornea — verify mapping)
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id,
    l."Is the patient on chelation treatment ?"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Is the patient on chelation treatment ?" = 'Yes';
""")

# Hydroxyurea treatment
execute("""
INSERT INTO omop.procedure_occurrence (
    procedure_occurrence_id, person_id, procedure_concept_id,
    procedure_date, procedure_datetime,
    procedure_end_date, procedure_end_datetime,
    procedure_type_concept_id, visit_occurrence_id, procedure_source_value
)
SELECT nextval('procedure_id_seq'), p.person_id,
    3169902,  -- Hydroxyurea therapy
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id,
    l."Is the patient on hydroxyurea treatment (present time) ?"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Is the patient on hydroxyurea treatment (present time) ?" = 'Yes';
""")

# Splenectomy
execute("""
INSERT INTO omop.procedure_occurrence (
    procedure_occurrence_id, person_id, procedure_concept_id,
    procedure_date, procedure_datetime,
    procedure_end_date, procedure_end_datetime,
    procedure_type_concept_id, visit_occurrence_id, procedure_source_value
)
SELECT nextval('procedure_id_seq'), p.person_id,
    2834904,  -- Removal of spleen (Lymphatic and Hemic Systems)
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id,
    l."Has the spleen been removed ?"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Has the spleen been removed ?" = 'Yes, totally';
""")

query("SELECT * FROM omop.procedure_occurrence")

## Final Overview

In [ ]:
query("""
SELECT table_schema, table_name,
       (xpath('/row/cnt/text()',
              query_to_xml(format('SELECT COUNT(*) AS cnt FROM %I.%I', table_schema, table_name),
                           false, true, '')))[1]::text::integer AS row_count
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
  AND table_type = 'BASE TABLE'
ORDER BY table_schema, table_name
""")